# DPN Classification — YOLOv11 Training (Google Colab)

This notebook trains the YOLOv11 model on a **free GPU** provided by Google Colab.
Training that takes 3+ hours on your CPU will finish in ~15 minutes here.

Your dataset is already in Google Drive (**ThermoDataBase** folder) — no upload needed.

## Before running:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU → Save
2. Run all cells top to bottom

## After training:
- `best_yolo_model.pt` is automatically saved to **MyDrive/DPN_Checkpoints/**
- Download it and place it in `checkpoints/` on your local machine
- Restart the API: `uvicorn api.main:app --reload`

## Step 1: Connect Google Drive and verify GPU

Your dataset is already in Google Drive in the folder **ThermoDataBase**.

Make sure the structure inside it looks like this:
```
My Drive/
  ThermoDataBase/
    Control Group/
      CG001_M/
        CG001_M_L.png
        CG001_M_L.csv
        ...
    DM Group/
      DM001_M/
        ...
```
If `Control Group/` and `DM Group/` are directly inside `ThermoDataBase/`, you're good to go.

In [1]:
# Verify GPU is available — must show a GPU, NOT 'No GPU'
import torch
if torch.cuda.is_available():
    print(f'GPU available: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected!')
    print('Go to Runtime -> Change runtime type -> T4 GPU -> Save, then re-run')

GPU available: Tesla T4
VRAM: 15.6 GB


In [2]:
# Mount Google Drive — a permissions popup will appear, click Allow
from google.colab import drive
drive.mount('/content/drive')

# Verify dataset is visible
import os
from pathlib import Path

DRIVE_DATASET = '/content/drive/MyDrive/ThermoDataBase'

if Path(DRIVE_DATASET).exists():
    control_dir = Path(DRIVE_DATASET) / 'Control Group'
    dm_dir      = Path(DRIVE_DATASET) / 'DM Group'
    control_count = len([d for d in control_dir.iterdir() if d.is_dir()]) if control_dir.exists() else 0
    dm_count      = len([d for d in dm_dir.iterdir()      if d.is_dir()]) if dm_dir.exists() else 0
    print(f'Dataset found: {DRIVE_DATASET}')
    print(f'  Control Group : {control_count} subjects')
    print(f'  DM Group      : {dm_count} subjects')
    if control_count == 0 or dm_count == 0:
        print()
        print('WARNING: One or both groups appear empty.')
        print('Check that Control Group/ and DM Group/ are directly inside ThermoDataBase/')
else:
    print(f'ERROR: Folder not found at {DRIVE_DATASET}')
    print('Make sure the folder is named exactly "ThermoDataBase" in your Google Drive root (My Drive)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ERROR: Folder not found at /content/drive/MyDrive/ThermoDataBase
Make sure the folder is named exactly "ThermoDataBase" in your Google Drive root (My Drive)


## Step 2: Install dependencies and clone your project

In [3]:
# Install required packages
# PyTorch is pre-installed on Colab with CUDA support — no need to reinstall
!pip install ultralytics>=8.3.0 scikit-learn>=1.4.0 scipy>=1.12.0 pyyaml>=6.0.0 --quiet
print('Packages installed.')

Packages installed.


In [ ]:
import sys, os, subprocess
from pathlib import Path

# Step 1 — Clone the project repo
GITHUB_REPO = 'https://github.com/catnipp9/transistors-thermal-ai-testing.git'
PROJECT_DIR = '/content/dpn_project'

if Path(PROJECT_DIR).exists():
    import shutil
    shutil.rmtree(PROJECT_DIR)

result = subprocess.run(['git', 'clone', GITHUB_REPO, PROJECT_DIR],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Git clone FAILED:')
    print(result.stderr)
    raise RuntimeError('Could not clone repo — check the URL above.')

# Step 2 — Add to Python path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Step 3 — Verify imports work
from models.data_loader import prepare_yolo_dataset
from models.model import YOLOv11DPNClassifier
from models.trainer import YOLOTrainer

print('Repo cloned and all modules loaded successfully.')

## Step 3: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Paths
DATA_DIR       = DRIVE_DATASET
CHECKPOINT_DIR = '/content/drive/MyDrive/DPN_Checkpoints'
YOLO_DATASET   = '/content/yolo_dataset'

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Data dir      : {DATA_DIR}')
print(f'Checkpoint dir: {CHECKPOINT_DIR}')
print(f'PyTorch       : {torch.__version__}')
print(f'CUDA          : {torch.cuda.is_available()}')

## Step 4: Build YOLO Dataset

Converts your thermogram PNGs into YOLO classification format with oversampling to fix class imbalance.

In [ ]:
# Build the YOLO dataset structure with oversampling
# This copies PNGs into train/val/test folders that YOLO expects
yaml_path = prepare_yolo_dataset(
    data_dir=DATA_DIR,
    output_dir=YOLO_DATASET,
)
print(f'Dataset ready: {yaml_path}')

## Step 5: Train YOLOv11

**Training settings — edit these to improve accuracy:**

| Setting | Current | Options |
|---------|---------|--------|
| `YOLO_VARIANT` | `yolo11l-cls` | `yolo11s/m/l/x-cls` |
| `epochs` | `100` | Higher = more training |
| `patience` | `20` | Higher = less early stopping |
| `imgsz` | `224` | `320` for more detail |

On a Colab T4 GPU, 100 epochs of `yolo11l-cls` takes ~15 minutes.

In [ ]:
# ── Training settings ──────────────────────────────────────────────────────
YOLO_VARIANT = 'yolo11l-cls'   # large model — good accuracy
EPOCHS       = 100
PATIENCE     = 20
IMGSZ        = 224
BATCH        = 32              # can use larger batch on GPU (was 16 on CPU)
LR           = 0.001
# ──────────────────────────────────────────────────────────────────────────

print(f'Model variant : {YOLO_VARIANT}')
print(f'Epochs        : {EPOCHS} (patience={PATIENCE})')
print(f'Image size    : {IMGSZ}x{IMGSZ}')
print(f'Batch size    : {BATCH}')
print(f'Device        : {"GPU - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('Starting training...')

yolo_model   = YOLOv11DPNClassifier(variant=YOLO_VARIANT)
yolo_trainer = YOLOTrainer(yolo_model, save_dir=CHECKPOINT_DIR)

yolo_history = yolo_trainer.train(
    data_yaml=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR,
    patience=PATIENCE,
)

print('Training complete!')
print(f'Results: {yolo_history}')

## Step 6: Save the trained model

In [ ]:
# Save best weights to Google Drive so you don't lose them when Colab resets
yolo_trainer.save_best_checkpoint()

saved_path = Path(CHECKPOINT_DIR) / 'best_yolo_model.pt'
if saved_path.exists():
    size_mb = saved_path.stat().st_size / 1e6
    print(f'Model saved to Google Drive: {saved_path}')
    print(f'File size: {size_mb:.1f} MB')
    print()
    print('Next steps:')
    print('  1. Download best_yolo_model.pt from the Files panel (left sidebar)')
    print('     OR find it in MyDrive/DPN_Checkpoints/ on Google Drive')
    print('  2. Place it in checkpoints/ on your local machine')
    print('  3. Restart the API: uvicorn api.main:app --reload')
else:
    print('WARNING: best_yolo_model.pt not found — check training completed successfully')

## Step 7: Quick accuracy test

In [ ]:
# Test the saved model on sample images from both groups
from api.inference import DPNClassifier

yolo_clf = DPNClassifier(
    model_path=str(Path(CHECKPOINT_DIR) / 'best_yolo_model.pt'),
    model_type='yolo',
)

control_imgs  = list((Path(DATA_DIR) / 'Control Group').glob('*/*_L.png'))[:5]
diabetic_imgs = list((Path(DATA_DIR) / 'DM Group').glob('*/*_L.png'))[:5]

correct = 0
total   = 0

print('Control group samples:')
for img in control_imgs:
    r  = yolo_clf.predict(str(img))
    ok = r['prediction'] == 'Control'
    correct += ok
    total   += 1
    print(f'  [{"v" if ok else "x"}] {img.name}: {r["prediction"]} ({r["confidence"]:.1f}%)')

print('\nDiabetic group samples:')
for img in diabetic_imgs:
    r  = yolo_clf.predict(str(img))
    ok = r['prediction'] == 'Diabetic'
    correct += ok
    total   += 1
    print(f'  [{"v" if ok else "x"}] {img.name}: {r["prediction"]} ({r["confidence"]:.1f}%)')

print(f'\nSample accuracy: {correct}/{total} = {correct/total*100:.1f}%')

## Step 8: Download the model to your PC

Run the cell below to download `best_yolo_model.pt` directly to your PC.
Then place it in `checkpoints/` in your project folder.

In [ ]:
# Download model file directly to your PC
from google.colab import files

model_path = Path(CHECKPOINT_DIR) / 'best_yolo_model.pt'
if model_path.exists():
    files.download(str(model_path))
    print('Download started — check your browser downloads folder.')
    print('Place the file in: checkpoints/best_yolo_model.pt')
else:
    print('Model file not found — make sure Step 6 ran successfully.')